In [3]:
import pandas as pd
from sklearn.cluster import KMeans 
from sklearn.mixture import  GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score




In [4]:
df = pd.read_csv("raw_wholesale_customers.csv")
df.head()

,Channel,Region,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen
0,2,3,12669,9656,7561,214,2674,1338
1,2,3,7057,9810,9568,1762,3293,1776
2,2,3,6353,8808,7684,2405,3516,7844
3,1,3,13265,1196,4221,6404,507,1788
4,2,3,22615,5410,7198,3915,1777,5185


In [5]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 440 entries, 0 to 439
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Channel           440 non-null    int64
 1   Region            440 non-null    int64
 2   Fresh             440 non-null    int64
 3   Milk              440 non-null    int64
 4   Grocery           440 non-null    int64
 5   Frozen            440 non-null    int64
 6   Detergents_Paper  440 non-null    int64
 7   Delicassen        440 non-null    int64
dtypes: int64(8)
memory usage: 27.6 KB
None


In [6]:
FEATURES = ["Fresh" , "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]


In [7]:
x = df[FEATURES].copy()

In [8]:
def iqr_fun(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower , upper

In [9]:
low_fresh,  high_fresh = iqr_fun(x["Fresh"])
low_milk,    high_milk = iqr_fun(x["Milk"])
low_grocery, high_grocery = iqr_fun(x["Grocery"])
low_frozen,  high_frozen = iqr_fun(x["Frozen"])
low_det,     high_det = iqr_fun(x["Detergents_Paper"])
low_deli,    high_deli = iqr_fun(x["Delicassen"])

In [10]:
x["Fresh"] = x["Fresh"].clip(lower=low_fresh,  upper=high_fresh)
x["Milk"] = x["Milk"].clip(lower=low_milk,    upper=high_milk)
x["Grocery"] = x["Grocery"].clip(lower=low_grocery, upper=high_grocery)
x["Frozen"] = x["Frozen"].clip(lower=low_frozen,  upper=high_frozen)
x["Detergents_Paper"] = x["Detergents_Paper"].clip(
    lower=low_det, upper=high_det)
x["Delicassen"] = x["Delicassen"].clip(lower=low_deli, upper=high_deli)

In [11]:
df[FEATURES] = x

In [12]:
print("\n=== IQR CAP SUMMARY  ===")
print(
    f"Fresh             low={low_fresh:.2f}  high={high_fresh:.2f}  |  after cap min={x['Fresh'].min():.2f}  max={x['Fresh'].max():.2f}")
print(
    f"Milk              low={low_milk:.2f}  high={high_milk:.2f}  |  after cap min={x['Milk'].min():.2f}  max={x['Milk'].max():.2f}")
print(
    f"Grocery           low={low_grocery:.2f}  high={high_grocery:.2f}  |  after cap min={x['Grocery'].min():.2f}  max={x['Grocery'].max():.2f}")
print(
    f"Frozen            low={low_frozen:.2f}  high={high_frozen:.2f}  |  after cap min={x['Frozen'].min():.2f}  max={x['Frozen'].max():.2f}")
print(
    f"Detergents_Paper  low={low_det:.2f}  high={high_det:.2f}  |  after cap min={x['Detergents_Paper'].min():.2f}  max={x['Detergents_Paper'].max():.2f}")
print(
    f"Delicassen        low={low_deli:.2f}  high={high_deli:.2f}  |  after cap min={x['Delicassen'].min():.2f}  max={x['Delicassen'].max():.2f}")

print("\n=== after iqr ===")
print(x.head())


=== IQR CAP SUMMARY  ===
Fresh             low=-17581.25  high=37642.75  |  after cap min=3.00  max=37642.75
Milk              low=-6952.88  high=15676.12  |  after cap min=55.00  max=15676.12
Grocery           low=-10601.12  high=23409.88  |  after cap min=3.00  max=23409.88
Frozen            low=-3475.75  high=7772.25  |  after cap min=25.00  max=7772.25
Detergents_Paper  low=-5241.12  high=9419.88  |  after cap min=3.00  max=9419.88
Delicassen        low=-1709.75  high=3938.25  |  after cap min=3.00  max=3938.25

=== after iqr ===
     Fresh    Milk  Grocery  Frozen  Detergents_Paper  Delicassen
0  12669.0  9656.0   7561.0   214.0            2674.0     1338.00
1   7057.0  9810.0   9568.0  1762.0            3293.0     1776.00
2   6353.0  8808.0   7684.0  2405.0            3516.0     3938.25
3  13265.0  1196.0   4221.0  6404.0             507.0     1788.00
4  22615.0  5410.0   7198.0  3915.0            1777.0     3938.25


In [13]:
scale = StandardScaler()
X_scaled = scale.fit_transform(x)
print("\nScaled shape:", X_scaled.shape)


Scaled shape: (440, 6)


In [14]:
k = 1
print("\n=== ELBOW METHOD ===")
for k in range(1, 11):
    km = KMeans(n_clusters=k, n_init="auto", random_state=42)
    km.fit(X_scaled)
    print(f"k={k} → SSE={km.inertia_:.2f}")


=== ELBOW METHOD ===
k=1 → SSE=2640.00
k=2 → SSE=1728.19
k=3 → SSE=1363.45
k=4 → SSE=1202.41
k=5 → SSE=1070.15
k=6 → SSE=964.76
k=7 → SSE=921.14
k=8 → SSE=776.63
k=9 → SSE=726.88
k=10 → SSE=707.41


In [15]:
kmeans = KMeans(n_clusters=k, n_init="auto", random_state=42)
labels = kmeans.fit_predict(X_scaled)

df["Cluster"] = labels.astype(int)
print("\n=== CLUSTERS ===")
print(df.head())


=== CLUSTERS ===
   Channel  Region    Fresh    Milk  Grocery  Frozen  Detergents_Paper  \
0        2       3  12669.0  9656.0   7561.0   214.0            2674.0   
1        2       3   7057.0  9810.0   9568.0  1762.0            3293.0   
2        2       3   6353.0  8808.0   7684.0  2405.0            3516.0   
3        1       3  13265.0  1196.0   4221.0  6404.0             507.0   
4        2       3  22615.0  5410.0   7198.0  3915.0            1777.0   

   Delicassen  Cluster  
0     1338.00        0  
1     1776.00        0  
2     3938.25        3  
3     1788.00        7  
4     3938.25        3  


In [16]:
sil = silhouette_score(X_scaled, labels)
dbi = davies_bouldin_score(X_scaled, labels)
print("\n=== metrics ===")
print(f"Silhouette Score : {sil:.3f} (closer to +1 is better)")
print(f"Davies–Bouldin   : {dbi:.3f} (lower is better)")


=== metrics ===
Silhouette Score : 0.241 (closer to +1 is better)
Davies–Bouldin   : 1.312 (lower is better)


In [17]:
centers_scaled = kmeans.cluster_centers_
centers_original = scale.inverse_transform(centers_scaled)

centers_df = pd.DataFrame(centers_original, columns=FEATURES)
centers_df.index.name = "Cluster"

print("\n=== CLUSTER CENTERS ===")
print(centers_df.round(2))


=== CLUSTER CENTERS ===
            Fresh      Milk   Grocery   Frozen  Detergents_Paper  Delicassen
Cluster                                                                     
0         5049.68   6274.58   8648.02  1074.62           3601.70      822.11
1         6834.10   1966.32   2552.48  1469.77            493.79      677.43
2        23042.12  14533.26  14349.07  6183.54           2363.24     3712.44
3        11728.17   6222.69   9173.55  1432.26           2921.69     2785.02
4        32791.73   4253.21   4972.12  6059.69            816.50     2420.83
5         9585.69   2398.23   2951.80  6596.26            669.34      618.03
6         4537.75   9281.73  16630.56  1049.35           7398.17      708.90
7        12785.14   3706.33   4007.24  7035.79            539.38     2075.90
8         8146.00  13787.83  22174.49  1900.13           8738.01     2184.92
9        25440.12   2893.12   3882.17  1882.40            660.67      907.94


In [18]:
GM_model =  GaussianMixture(n_components=5)
df['GM_cluster'] = GM_model.fit_predict(X_scaled)
print("\n GMM is successfuly predcted")


 GMM is successfuly predcted


In [19]:
GMM_silhoute = silhouette_score(X_scaled, df['GM_cluster'])
print(f"\n GMM silhouete score: {GMM_silhoute:.4f}")
if sil > GMM_silhoute:
    print("kmeans is better than gmm")
else:
    print("GMM is better than kmeans")
                                


 GMM silhouete score: 0.1576
kmeans is better than gmm


In [21]:
sample_idx = [0, 1, 2]
sanity = df.loc[sample_idx, ["Channel", "Region"] + FEATURES + ["Cluster"]]
print("\n=== SANITY CHECK  ===")
print(sanity)
                                


=== SANITY CHECK  ===
   Channel  Region    Fresh    Milk  Grocery  Frozen  Detergents_Paper  \
0        2       3  12669.0  9656.0   7561.0   214.0            2674.0   
1        2       3   7057.0  9810.0   9568.0  1762.0            3293.0   
2        2       3   6353.0  8808.0   7684.0  2405.0            3516.0   

   Delicassen  Cluster  
0     1338.00        0  
1     1776.00        0  
2     3938.25        3  


In [23]:
OUT_PATH = "wholesale_customers.csv"
df.to_csv(OUT_PATH, index=False)
print(f"\nSaved clustered dataset → {OUT_PATH}")


Saved clustered dataset → wholesale_customers.csv
